# Laboratorio 7: Pipeline ETL, Profiling y Gobernanza en el Data Lake
**Materia:** Fundamentos de Big Data (IDSD413) - EPN

**Objetivo:** Implementar un pipeline ETL para procesar 5 millones de transacciones, aplicando perfilamiento, eliminación de duplicados, descarte inteligente de registros nulos, imputación por mediana y detección de valores atípicos mediante IQR.

Se implementa un Data Lake con tres capas: Raw, que conserva los datos originales; Quarantine, que almacena los outliers para auditoría; y Curated, que contiene los datos limpios, normalizados y estandarizados para su posterior análisis.


In [1]:
import pandas as pd
import numpy as np
import os

print("1/3 Generando datos base perfectos en memoria...")
np.random.seed(42)
N_ROWS = 5_000_000
df_base = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

# Guardamos una base fresca para evitar el error de bytes corruptos en Colab
nombre_fresco = 'base_fresca.parquet'
df_base.to_parquet(nombre_fresco, engine='pyarrow')

print("2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...")
df_original = pd.read_parquet(nombre_fresco, engine='pyarrow')

# a) Inyectamos un 5% de nulos
n_nulos = int(len(df_original) * 0.05)
indices_nulos = np.random.choice(df_original.index, n_nulos, replace=False)
df_original.loc[indices_nulos, 'monto'] = np.nan

# b) Duplicamos 50,000 registros
df_duplicados = df_original.sample(n=50000)
df_ingesta = pd.concat([df_original, df_duplicados], ignore_index=True)

# c) Inyectamos outliers (valores absurdos que rompen la estadística)
df_ingesta.loc[0:100, 'monto'] = 9999999.99

print(f"3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: {len(df_ingesta):,}")

1/3 Generando datos base perfectos en memoria...
2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...
3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: 5,050,000


## Fase 1: Profiling y Capa RAW (El Diagnóstico del Caos)
Antes de transformar, debemos perfilar el archivo crudo para entender la magnitud del problema y luego guardarlo en la capa inmutable del Data Lake.

In [2]:
print("--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---\n")

# 1. Auditoría de Nulos y Duplicados
print("1. Valores Nulos por Columna:")
print(df_ingesta.isnull().sum())
print(f"\n2. Registros Duplicados Exactos: {df_ingesta.duplicated().sum():,}")

# 2. Auditoría de Outliers (Fíjense en la diferencia entre el promedio y el max)
print("\n3. Resumen Estadístico de 'monto':")
print(df_ingesta['monto'].describe().apply(lambda x: f"{x:,.2f}"))

# 3. Guardado en Capa RAW (Inmutable)
os.makedirs('./datalake/raw', exist_ok=True)
os.makedirs('./datalake/quarantine', exist_ok=True)
os.makedirs('./datalake/curated', exist_ok=True)

print("\n--- EXTRACT & LOAD (EL) ---")
print("Guardando evidencia en la Capa RAW...")
df_ingesta.to_parquet('./datalake/raw/ingesta_transacciones_sucio.parquet')
print("¡Archivo RAW almacenado con éxito! Listos para limpiar.")

--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---

1. Valores Nulos por Columna:
id_transaccion         0
id_cliente             0
monto             252510
categoria              0
estado                 0
dtype: int64

2. Registros Duplicados Exactos: 49,999

3. Resumen Estadístico de 'monto':
count    4,797,490.00
mean         1,463.00
std         45,882.62
min              5.00
25%            628.91
50%          1,252.56
75%          1,876.37
max      9,999,999.99
Name: monto, dtype: object

--- EXTRACT & LOAD (EL) ---
Guardando evidencia en la Capa RAW...
¡Archivo RAW almacenado con éxito! Listos para limpiar.


## Fase 2: Transform (T) - Pipeline ETL de Limpieza y Normalización
Durante la transformación se eliminan registros duplicados y transacciones canceladas con montos nulos. Para los demás estados se aplica imputación mediante la mediana por categoría. Los valores atípicos se identifican mediante el rango intercuartílico (IQR) y se conservan en la capa Quarantine en formato Parquet. Finalmente, se aplican normalización Min-Max y estandarización Z-score.

In [3]:
print("Iniciando Transformación...\n")

# 1. Eliminación de Duplicados
antes = len(df_ingesta)
df_limpio = df_ingesta.drop_duplicates().copy()
print(f"Duplicados eliminados: {antes - len(df_limpio):,}")

# Reto 2
# 2. Tratamiento de Nulos - Descarte Inteligente

# Identificar transacciones canceladas con monto nulo
condicion_descarte = (
    (df_limpio['estado'] == 'Cancelado') &
    (df_limpio['monto'].isna())
)

# Contabilizar registros que se eliminarán
cancelados_eliminados = condicion_descarte.sum()

# Eliminar únicamente las filas que cumplen la condición
df_limpio = df_limpio.loc[~condicion_descarte].copy()

# Imputar la mediana por categoría al resto
medianas = df_limpio.groupby('categoria')['monto'].transform('median')

df_limpio['monto'] = df_limpio['monto'].fillna(medianas)

# Verificar resultados
print(f"Cancelados con monto nulo eliminados: {cancelados_eliminados:,}")
print(f"Nulos restantes en monto: {df_limpio['monto'].isna().sum():,}")

# Reto 1
# 3. Tratamiento de Outliers (Regla IQR)
Q1 = df_limpio['monto'].quantile(0.25)
Q3 = df_limpio['monto'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Detectar outliers inferiores y superiores
mascara_outliers = (
    (df_limpio['monto'] < limite_inferior) |
    (df_limpio['monto'] > limite_superior)
)

df_outliers = df_limpio.loc[mascara_outliers].copy()

# Guardar los outliers en QUARANTINE
df_outliers.to_parquet(
    './datalake/quarantine/outliers.parquet',
    engine='pyarrow',
    index=False
)

# Mantener los registros válidos
df_limpio = df_limpio.loc[~mascara_outliers].copy()

print(f"Outliers enviados a cuarentena: {len(df_outliers):,}")
print("Archivo Parquet guardado en QUARANTINE correctamente.")


# 4. Normalización Min-Max (Escala de 0 a 1)
min_monto = df_limpio['monto'].min()
max_monto = df_limpio['monto'].max()
df_limpio['monto_escalado'] = (df_limpio['monto'] - min_monto) / (max_monto - min_monto)
print("Normalización Min-Max completada.")
# 5. Estandarización estadística Z-score

media_monto = df_limpio['monto'].mean()
desviacion_monto = df_limpio['monto'].std()

if desviacion_monto > 0:
    df_limpio['monto_zscore'] = (
        (df_limpio['monto'] - media_monto)
        / desviacion_monto
    )
else:
    df_limpio['monto_zscore'] = 0.0

print("Estandarización Z-score completada.")

Iniciando Transformación...

Duplicados eliminados: 49,999
Cancelados con monto nulo eliminados: 62,513
Nulos restantes en monto: 0
Outliers enviados a cuarentena: 101
Archivo Parquet guardado en QUARANTINE correctamente.
Normalización Min-Max completada.
Estandarización Z-score completada.


In [4]:
# Verificación del Reto 2

print("=== VERIFICACIÓN DE LIMPIEZA ===")

print("Cancelados con monto nulo eliminados:",
      cancelados_eliminados)

print("Nulos restantes en monto:",
      df_limpio['monto'].isna().sum())

print("Registros válidos después de limpieza:",
      len(df_limpio))

print("Outliers almacenados en cuarentena:",
      len(df_outliers))

# Validaciones automáticas
assert df_limpio['monto'].isna().sum() == 0
assert len(df_outliers) == 101

print("\nVALIDACIONES COMPLETADAS CORRECTAMENTE")

=== VERIFICACIÓN DE LIMPIEZA ===
Cancelados con monto nulo eliminados: 62513
Nulos restantes en monto: 0
Registros válidos después de limpieza: 4937387
Outliers almacenados en cuarentena: 101

VALIDACIONES COMPLETADAS CORRECTAMENTE


In [5]:
import os

print("Carpetas del Data Lake:")
print(os.listdir('./datalake'))

print("\nArchivos en cuarentena:")
print(os.listdir('./datalake/quarantine'))

print("\nCantidad de outliers:")
print(len(df_outliers))

Carpetas del Data Lake:
['raw', 'quarantine', 'curated']

Archivos en cuarentena:
['outliers.parquet']

Cantidad de outliers:
101


## Fase 3: Load (Curated Layer) y Verificación Final
Guardamos el dato limpio en formato columnar en la capa *Curated*. Finalmente, comparamos la estadística del dato curado frente al desastre original.

In [6]:
# 1. Guardamos en la capa analítica
df_limpio.to_parquet('./datalake/curated/transacciones_model_ready.parquet')
print("Datos escritos en la Capa CURATED exitosamente.\n")

# 2. Demostración del dato limpio (El "Después")
print("--- ESTADÍSTICA DEL DATO CURADO (Monto limpio vs Escalado) ---")
# Comparamos la estadística final para demostrar que la media volvió a la normalidad
print(df_limpio[['monto', 'monto_escalado', 'monto_zscore']].describe().applymap(lambda x: f"{x:,.2f}"))

print("\n--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---")
display(df_limpio[['id_transaccion', 'categoria', 'monto', 'monto_escalado']].head(5))

print("\n--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---")
print(f"RAW (Sucio): {os.path.getsize('./datalake/raw/ingesta_transacciones_sucio.parquet') / (1024*1024):.2f} MB")
print(f"CURATED (Limpio): {os.path.getsize('./datalake/curated/transacciones_model_ready.parquet') / (1024*1024):.2f} MB")

Datos escritos en la Capa CURATED exitosamente.

--- ESTADÍSTICA DEL DATO CURADO (Monto limpio vs Escalado) ---
              monto monto_escalado  monto_zscore
count  4,937,387.00   4,937,387.00  4,937,387.00
mean       1,252.50           0.50         -0.00
std          706.46           0.28          1.00
min            5.00           0.00         -1.77
25%          653.39           0.26         -0.85
50%        1,252.53           0.50          0.00
75%        1,851.88           0.74          0.85
max        2,500.00           1.00          1.77

--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---


/tmp/ipykernel_15670/1882778251.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(df_limpio[['monto', 'monto_escalado', 'monto_zscore']].describe().applymap(lambda x: f"{x:,.2f}"))


,id_transaccion,categoria,monto,monto_escalado
101,101,Juguetes,1015.580233,0.405042
102,102,Alimentos,1837.507104,0.734472
103,103,Electronica,997.149883,0.397655
104,104,Ropa,2021.055379,0.808038
105,105,Juguetes,1251.907721,0.499762



--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---
RAW (Sucio): 74.41 MB
CURATED (Limpio): 171.89 MB


In [7]:
import os
import pandas as pd

print("=== ESTRUCTURA FINAL DEL DATA LAKE ===")

for carpeta in ['raw', 'quarantine', 'curated']:
    ruta = f'./datalake/{carpeta}'
    print(f"\n{carpeta.upper()}:")
    print(os.listdir(ruta))

print("\n=== VERIFICACIÓN DE CUARENTENA ===")

df_verificacion = pd.read_parquet(
    './datalake/quarantine/outliers.parquet'
)

print("Outliers conservados:", len(df_verificacion))

print("\n=== VERIFICACIÓN DE DATOS LIMPIOS ===")

print("Registros procesados:", len(df_limpio))
print("Montos nulos restantes:",
      df_limpio['monto'].isna().sum())

=== ESTRUCTURA FINAL DEL DATA LAKE ===

RAW:
['ingesta_transacciones_sucio.parquet']

QUARANTINE:
['outliers.parquet']

CURATED:
['transacciones_model_ready.parquet']

=== VERIFICACIÓN DE CUARENTENA ===
Outliers conservados: 101

=== VERIFICACIÓN DE DATOS LIMPIOS ===
Registros procesados: 4937387
Montos nulos restantes: 0


In [8]:
df_final = pd.read_parquet(
    './datalake/curated/transacciones_model_ready.parquet'
)

print("Columnas del archivo final:")
print(df_final.columns.tolist())

print("\nEstadísticas Z-score:")
print(df_final['monto_zscore'].describe())

Columnas del archivo final:
['id_transaccion', 'id_cliente', 'monto', 'categoria', 'estado', 'monto_escalado', 'monto_zscore']

Estadísticas Z-score:
count    4.937387e+06
mean    -1.827896e-16
std      1.000000e+00
min     -1.765845e+00
25%     -8.480443e-01
50%      3.795470e-05
75%      8.484179e-01
max      1.765843e+00
Name: monto_zscore, dtype: float64
